# OpenCode + DeepSeek-V4-Pro 9B — TODO EN UNO

**Único requisito:** Settings (panel derecho) → **Accelerator: GPU T4 x2** · **Internet: ON** → **Run All**.

Qué hace (automático, ~25-30 min la primera vez):
1. Instala vLLM y sirve el modelo 9B en `localhost:8000` (API OpenAI-compatible)
2. Restaura la workstation de **opencode-continuum** (Dataset) y levanta **OpenCode Web** en el puerto 4096
3. Inyecta el provider del 9B en el config de opencode y reinicia el proceso (el watchdog de continuum lo relanza)
4. Imprime la **URL trycloudflare** que abre opencode en tu navegador/teléfono → ahí seleccionás el modelo **DeepSeek-V4-Pro-Qwen3.5-9B (heretic)**

⚠️ La URL cambia cada reinicio del kernel. Los secretos opcionales (NVIDIA_API_KEY, OPENCODE_SERVER_PASSWORD, OPENCODE_CLOUD_DATASET) se leen de Kaggle Secrets como en continuum.

In [ ]:
# 1) Dependencia del servidor de modelo
!pip install -q vllm

In [ ]:
# 2) Servir el modelo 9B (reparte entre las 2 T4) y esperar a que la API responda
import subprocess, time, urllib.request, os

MODEL = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"
comando = [
    "vllm", "serve", MODEL,
    "--tensor-parallel-size", "2",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.92",
    "--port", "8000",
]
vllm_log = open("/kaggle/working/vllm.log", "w")
subprocess.Popen(comando, stdout=vllm_log, stderr=subprocess.STDOUT)
print("vLLM arrancando (descarga ~18 GB la primera vez)...", flush=True)

listo = False
for i in range(60):                     # hasta 30 min
    time.sleep(30)
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"API del modelo lista tras {(i+1)*30} s")
        listo = True
        break
    except Exception:
        print(f"  [{(i+1)*30}s] cargando modelo...", flush=True)

if not listo:
    print("TIMEOUT. Últimas líneas de vllm.log:")
    print("\n".join(open("/kaggle/working/vllm.log").readlines()[-15:]))
    raise RuntimeError("vLLM no respondió en 30 min")

In [ ]:
# 3) Continuum: restaurar workstation + levantar OpenCode Web + túneles + links
#    (misma maquinaria de la celda v13 del repo, sin el keep-alive final)
import base64, json, os, re, shutil, socket, subprocess, sys, time
import urllib.parse, urllib.request
from datetime import datetime, timezone
from pathlib import Path

def log(*a, **k):
    print(*a, flush=True, **k)

def run(cmd, check=True, **kw):
    log(" $", " ".join(str(c) for c in cmd))
    return subprocess.run(cmd, check=check, **kw)

REPO = Path("/kaggle/working/opencode-continuum")
WORKSPACE = Path("/kaggle/working/opencode_cloud/workspace")
METADATA = Path("/kaggle/working/opencode_cloud/metadata")
LOGS = Path("/kaggle/working/opencode_cloud/logs")
PORT = 4096
PICKER_PORT = 4097
DATASET_ID = os.environ.get("OPENCODE_CLOUD_DATASET", "marceloate/opencode-cloud-state")

log("=" * 60)
log("OpenCode Continuum + modelo 9B — workstation en una celda")
log("=" * 60)

# secrets opcionales
try:
    from kaggle_secrets import UserSecretsClient
    usc = UserSecretsClient()
    for key in ("NVIDIA_API_KEY", "OPENCODE_CLOUD_DATASET",
                "OPENCODE_SERVER_PASSWORD", "OPENCODE_SERVER_USERNAME",
                "CLOUDFLARE_TUNNEL_TOKEN"):
        try:
            val = usc.get_secret(key)
            if val:
                os.environ[key] = val
                if key == "OPENCODE_CLOUD_DATASET":
                    DATASET_ID = val
        except Exception:
            pass
except Exception as e:
    log("secrets:", type(e).__name__)

os.environ.setdefault("OPENCODE_CLOUD_DATASET", DATASET_ID)
log("Dataset:", DATASET_ID)

for mod, pkg in (("kagglehub", "kagglehub"), ("requests", "requests")):
    try:
        __import__(mod)
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "-q", pkg], timeout=300)

if not (REPO / "src/opencode_kaggle/bootstrap.py").exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    run(["git", "clone", "--depth", "1",
         "https://github.com/enrrutador/opencode-continuum.git", str(REPO)], timeout=180)
else:
    run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False, timeout=120)

sys.path.insert(0, str(REPO / "src"))

WORKSPACE.mkdir(parents=True, exist_ok=True)
if not (WORKSPACE / ".git").exists():
    run(["git", "init"], cwd=str(WORKSPACE), check=False)
    run(["git", "-C", str(WORKSPACE), "config", "user.email", "opencode@kaggle.local"], check=False)
    run(["git", "-C", str(WORKSPACE), "config", "user.name", "OpenCode"], check=False)
    (WORKSPACE / "README.md").write_text("# OpenCode workspace\n", encoding="utf-8")
    run(["git", "-C", str(WORKSPACE), "add", "-A"], check=False)
    run(["git", "-C", str(WORKSPACE), "commit", "-m", "init"], check=False)

log("\n[bootstrap]")
from opencode_cloud.checkpoint import CheckpointPolicy
from opencode_kaggle.bootstrap import bootstrap

policy = CheckpointPolicy(local_interval=120, min_publish_interval=900)
info = bootstrap(dataset_id=DATASET_ID, opencode_port=PORT, policy=policy, enable_access_layer=True)
log(json.dumps({k: v for k, v in info.items() if k != "shutdown"}, indent=2, default=str))

if not info.get("ok"):
    if info.get("status") == "OLD_SERVER_RUNNING":
        log("\n!! El servidor ya está corriendo de un bootstrap anterior: NO lo reinicies.")
    else:
        raise RuntimeError(f"Bootstrap falló: {info}")

recovery = info.get("recovery", "")

wa = info.get("web_access") or {}
base_url = (wa.get("url") or "").rstrip("/")
if not base_url:
    lp = LOGS / "cloudflare-tunnel.log"
    if lp.exists():
        urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", lp.read_text())
        if urls:
            base_url = urls[-1]
named = wa.get("provider") == "cloudflare_named_tunnel"

ws_b64 = base64.urlsafe_b64encode(str(WORKSPACE).encode()).decode().rstrip("=")

def api(method, path, data=None):
    url = f"http://127.0.0.1:{PORT}{path}"
    req = urllib.request.Request(url, data=data, method=method)
    req.add_header("Content-Type", "application/json")
    pw = os.environ.get("OPENCODE_SERVER_PASSWORD", "")
    user = os.environ.get("OPENCODE_SERVER_USERNAME", "opencode")
    if pw:
        tok = base64.b64encode(f"{user}:{pw}".encode()).decode()
        req.add_header("Authorization", f"Basic {tok}")
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode() or "null")

from opencode_cloud.entry import load_pin, pick_session, save_pin

pin_path = METADATA / "session.json"
q = "?directory=" + urllib.parse.quote(str(WORKSPACE))
sessions = []
try:
    sessions = api("GET", "/session" + q) or []
    if not isinstance(sessions, list):
        sessions = []
except Exception as e:
    log("GET /session:", type(e).__name__, e)

chosen = pick_session(sessions, load_pin(pin_path))
if chosen:
    sid_path = f"/{ws_b64}/session/{chosen['id']}"
    OPEN_TITLE = chosen.get("title") or chosen["id"]
    METADATA.mkdir(parents=True, exist_ok=True)
    save_pin(pin_path, chosen["id"], OPEN_TITLE)
else:
    sid_path = f"/{ws_b64}/session"
    OPEN_TITLE = "(nueva conversación)"
OPEN_URL = (base_url + sid_path) if base_url else sid_path

def picker_url_from_log():
    try:
        s = socket.create_connection(("127.0.0.1", PICKER_PORT), timeout=2)
        s.close()
    except OSError:
        return ""
    lp = LOGS / "cloudflare-picker-tunnel.log"
    if not lp.exists():
        return ""
    urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", lp.read_text())
    return urls[-1] if urls else ""

sp = info.get("session_picker") or {}
if sp.get("available") and sp.get("url"):
    PICKER_URL = sp["url"].rstrip("/")
elif named and base_url:
    PICKER_URL = base_url + "/go"
else:
    PICKER_URL = picker_url_from_log()

out = Path("/kaggle/working/opencode_cloud/OPEN_THIS_URL.txt")
lines = [
    f"ELEGIR SESION: {PICKER_URL}",
    f"ULTIMA SESION: {OPEN_URL}",
    "",
    f"sesion: {OPEN_TITLE}",
    f"recovery: {recovery}",
    f"sesiones: {len(sessions)}",
]
out.write_text("\n".join(lines) + "\n", encoding="utf-8")

log("\n######## ELEGIR SESIÓN ########")
if PICKER_URL:
    print(PICKER_URL, flush=True)
log("### o continuá directamente la última ###")
print(OPEN_URL, flush=True)
log("###############################")

In [ ]:
# 4) Inyectar el provider del 9B (vLLM local) y reiniciar OpenCode
#    El watchdog de continuum detecta la muerte del proceso y lo relanza con el config nuevo.
import json, pathlib, socket, subprocess, time

CFG = pathlib.Path("/kaggle/working/opencode_cloud/xdg/config/opencode.json")
cfg = json.loads(CFG.read_text(encoding="utf-8")) if CFG.exists() else {}
cfg.setdefault("provider", {})
cfg["provider"]["kaggle-deepseek"] = {
    "name": "DeepSeek-V4-Pro 9B (vLLM local)",
    "npm": "@ai-sdk/openai-compatible",
    "options": {"baseURL": "http://localhost:8000/v1"},
    "models": {
        "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic": {
            "name": "DeepSeek-V4-Pro-Qwen3.5-9B (heretic)"
        }
    },
}
CFG.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
print("Config parchado con provider kaggle-deepseek -> http://localhost:8000/v1")

subprocess.run(["pkill", "-f", "opencode web"], check=False)
print("OpenCode detenido; esperando relanzamiento del watchdog...")

vivo = False
for i in range(45):                    # hasta 90 s
    time.sleep(2)
    try:
        s = socket.create_connection(("127.0.0.1", 4096), timeout=2)
        s.close()
        vivo = True
        break
    except OSError:
        pass

if vivo:
    print(f"OpenCode reiniciado tras {(i+1)*2} s — modelo 9B ya seleccionable en la UI")
else:
    print("AVISO: el puerto 4096 no respondió en 90 s; revisá los logs de continuum.")

# reimprimir links (el túnel sobrevive al reinicio del proceso opencode)
try:
    contenido = (LOGS / "cloudflare-tunnel.log").read_text()
    urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", contenido)
    if urls:
        print("\nTu URL de opencode:")
        print(urls[-1] + (OPEN_URL[len(base_url):] if base_url and OPEN_URL.startswith(base_url) else ""))
except Exception:
    pass

In [ ]:
# 5) Keep-alive: mantiene la sesión viva (Interrumpir esta celda para frenar)
ticks = 0
while True:
    s = socket.socket()
    s.settimeout(5)
    try:
        s.connect(("127.0.0.1", PORT))
        alive = True
    except OSError:
        alive = False
    finally:
        s.close()
    ticks += 1
    keep_alive.ticks = ticks
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    if alive:
        if ticks % 10 == 1:
            log(f"[keep-alive] {now} OpenCode OK en {PORT} | modelo 9B en localhost:8000")
    else:
        log(f"[keep-alive] {now} puerto {PORT} sin respuesta")
    time.sleep(60)